In [1]:
import mpmath
print(mpmath.__version__)
mpmath.mp.dps = 50
# Test first 3 zeros
import time
t0 = time.time()
z1 = mpmath.zetazero(1)
z2 = mpmath.zetazero(2)
z3 = mpmath.zetazero(3)
print(time.time()-t0)
print(z1)
print(z2)
print(z3)


1.3.0
0.07163310050964355
(0.5 + 14.134725141734693790457251983562470270784257115699j)
(0.5 + 21.022039638771554992628479593896902777334340524903j)
(0.5 + 25.010857580145688763213790992562821818659549672558j)


In [2]:
# Verify match to R2 reference (9+ decimal places)
refs = {1: "14.134725141", 2: "21.022039639", 3: "25.010857580"}
for k, ref in refs.items():
 z = mpmath.zetazero(k)
 im_str = mpmath.nstr(z.imag, 15)
 print(f"γ_{k}: computed={im_str}, ref={ref}")
 # Check first 11 chars (digits + decimal)
 computed_rounded = mpmath.nstr(z.imag, 11)
 print(f" rounded(11 sig)={computed_rounded}")


γ_1: computed=14.1347251417347, ref=14.134725141
 rounded(11 sig)=14.134725142
γ_2: computed=21.0220396387716, ref=21.022039639
 rounded(11 sig)=21.022039639
γ_3: computed=25.0108575801457, ref=25.010857580
 rounded(11 sig)=25.01085758


In [3]:
# Decimal-place matching: reference truncated/rounded to 9 dp
# γ1: 14.134725141734693... vs 14.134725141 -> match to 9 dp (truncation)
# γ2: 21.022039638771... vs 21.022039639 -> 21.022039639 rounds from .638771... → 21.022039639 ✓ (9 dp rounded)
# γ3: 25.010857580145... vs 25.010857580 -> match to 9 dp ✓
# All match to 9 decimal places. R2 passes.

# Estimate timing on a small batch to extrapolate
import time
times = []
for n in [10, 50, 100]:
 t0 = time.time()
 _ = mpmath.zetazero(n)
 times.append((n, time.time()-t0))
print(times)


[(10, 0.029796123504638672), (50, 0.03862333297729492), (100, 0.20101213455200195)]


In [4]:
# Quick batch test - compute 100 zeros to estimate
import time
t0 = time.time()
sample = [mpmath.zetazero(n).imag for n in range(1, 101)]
elapsed_100 = time.time() - t0
print(f"100 zeros: {elapsed_100:.1f}s")
# Higher zeros take longer; let's check around n=1000, 5000
t0 = time.time()
_ = mpmath.zetazero(1000)
print(f"single zetazero(1000): {time.time()-t0:.2f}s")
t0 = time.time()
_ = mpmath.zetazero(5000)
print(f"single zetazero(5000): {time.time()-t0:.2f}s")


100 zeros: 8.4s


single zetazero(1000): 0.43s


single zetazero(5000): 1.00s


In [5]:
# Sample timings at various n to estimate total cost
import time
sample_ns = [500, 1000, 2000, 3000, 4000, 5000]
for n in sample_ns:
 t0 = time.time()
 _ = mpmath.zetazero(n)
 print(f"n={n}: {time.time()-t0:.3f}s")


n=500: 0.327s


n=1000: 0.440s


n=2000: 0.609s


n=3000: 0.800s


n=4000: 1.055s


n=5000: 0.976s


In [6]:
# Rough estimate: ~0.5s avg * 5000 = ~40 min. Feasible. Let's go.
# Compute all 5000 zeros, store imaginary parts as high-precision strings + as float64 array.
import time, os, sys, gc
import numpy as np
import psutil

process = psutil.Process(os.getpid())
mem_start = process.memory_info().rss / 1024**2
print(f"Memory start: {mem_start:.1f} MB")

mpmath.mp.dps = 50
N = 5000

zeros_mpf = [] # mpmath mpf (50 dps)
t0 = time.time()
last_report = t0
for n in range(1, N+1):
 z = mpmath.zetazero(n)
 zeros_mpf.append(z.imag)
 if n % 250 == 0:
 now = time.time()
 rate = n / (now - t0)
 eta = (N - n) / rate
 mem_now = process.memory_info().rss / 1024**2
 print(f"n={n}, elapsed={now-t0:.1f}s, rate={rate:.1f}/s, ETA={eta:.0f}s, mem={mem_now:.1f}MB")
 last_report = now

elapsed = time.time() - t0
mem_end = process.memory_info().rss / 1024**2
print(f"\nTotal: {elapsed:.1f}s ({elapsed/60:.2f} min)")
print(f"Memory end: {mem_end:.1f} MB (delta {mem_end-mem_start:.1f} MB)")


TimeoutError: Code execution timed out after 900.0 seconds

In [7]:
# Cell timed out at 900s. Check what we have in memory.
print(f"Zeros computed so far: {len(zeros_mpf)}")
print(f"Elapsed before timeout: ~900s")
if len(zeros_mpf) > 0:
 print(f"First zero: {zeros_mpf[0]}")
 print(f"Last zero: {zeros_mpf[-1]}")


Zeros computed so far: 2003
Elapsed before timeout: ~900s
First zero: 14.134725141734693790457251983562470270784257115699
Last zero: 2518.552852040690821002384122456039573660211620685


In [8]:
# Good - we have 2003 zeros so far. The cell timeout is 900s, but we can continue.
# Rate so far: ~2003/900 = 2.2/s. Remaining 2997 zeros * 0.45s = ~1350s. Need to chunk.
# Let's continue from n=2004 in chunks fitting under 900s.
import time
t0 = time.time()
start_n = len(zeros_mpf) + 1
# Aim for ~700s per chunk = ~1500 zeros
chunk_end = min(start_n + 1400, 5001)
for n in range(start_n, chunk_end):
 z = mpmath.zetazero(n)
 zeros_mpf.append(z.imag)
 if n % 200 == 0:
 elapsed = time.time() - t0
 print(f"n={n}, chunk elapsed={elapsed:.1f}s, total len={len(zeros_mpf)}")
print(f"\nChunk done. Total: {len(zeros_mpf)}, chunk time: {time.time()-t0:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [9]:
print(f"Total zeros now: {len(zeros_mpf)}")
print(f"Last: {zeros_mpf[-1]}")


Total zeros now: 3332
Last: 3860.0930085051726903187498578626310814507561620579


In [10]:
# Continue - 1668 left, at ~0.55s/each = ~917s. Chunk to ~1300.
import time
t0 = time.time()
start_n = len(zeros_mpf) + 1
chunk_end = min(start_n + 1300, 5001)
for n in range(start_n, chunk_end):
 z = mpmath.zetazero(n)
 zeros_mpf.append(z.imag)
 if n % 200 == 0:
 elapsed = time.time() - t0
 print(f"n={n}, chunk elapsed={elapsed:.1f}s")
print(f"\nDone. Total: {len(zeros_mpf)}, chunk time: {time.time()-t0:.1f}s")


TimeoutError: Code execution timed out after 900.0 seconds

In [11]:
print(f"Total zeros now: {len(zeros_mpf)}")
print(f"Last: {zeros_mpf[-1]}")
remaining = 5000 - len(zeros_mpf)
print(f"Remaining: {remaining}")


Total zeros now: 4285
Last: 4777.6356769832707110380527153779038267552061335619
Remaining: 715


In [12]:
# 715 left at ~0.65s = ~465s. Let's finish, but save progress first.
import numpy as np, os, json, time

# Save partial in case timeout
os.makedirs("/workspace/8480ed88-494a-4ef3-aa14-5929fd69adbb", exist_ok=True)
partial_strs = [mpmath.nstr(z, 50) for z in zeros_mpf]
with open("/workspace/8480ed88-494a-4ef3-aa14-5929fd69adbb/zeta_zeros_partial.json", "w") as f:
 json.dump(partial_strs, f)
print(f"Saved {len(partial_strs)} partial")

t0 = time.time()
start_n = len(zeros_mpf) + 1
for n in range(start_n, 5001):
 z = mpmath.zetazero(n)
 zeros_mpf.append(z.imag)
 if n % 200 == 0:
 print(f"n={n}, chunk elapsed={time.time()-t0:.1f}s")
print(f"\nFinal len: {len(zeros_mpf)}, chunk time: {time.time()-t0:.1f}s")


TimeoutError: Code execution timed out after 207.0 seconds